In [23]:
import sys
from pathlib import Path
root = Path().resolve().parent  # adjust level as needed
sys.path.insert(0, str(root))

In [2]:
from config import output_path_prefix

In [3]:
import pickle

In [28]:
output_dir = root / "outputs"
files = list(output_dir.glob("SPRI_20*_output_docs.pkl"))

In [29]:
files.sort()

In [30]:
files

[PosixPath('/home/jake/RAG-end-to-end/outputs/SPRI_2022_output_docs.pkl'),
 PosixPath('/home/jake/RAG-end-to-end/outputs/SPRI_2023_output_docs.pkl'),
 PosixPath('/home/jake/RAG-end-to-end/outputs/SPRI_2025_output_docs.pkl')]

In [31]:
docs = []
for file_path in files:
    print(f"Loading: {file_path.name}")
    with open(file_path, "rb") as f:
        doc = pickle.load(f)
    docs.extend(doc)

Loading: SPRI_2022_output_docs.pkl
Loading: SPRI_2023_output_docs.pkl
Loading: SPRI_2025_output_docs.pkl


In [4]:
with open(f"{output_path_prefix}_docs.pkl", "rb") as f:
    docs = pickle.load(f)

In [9]:
docs[0]

Document(metadata={'page': 1, 'image_id': [], 'image_path': []}, page_content='\n\nISSUE REPORT l 2025.04.15. IS-200\n\n\nA I Index 2025 주요 내용과 시사점\n\nSummary and Implications of 2025 AI Index Report\n\n안성원/임영모/유재흥/안미소/장진철/이해수/김지원/임정주\n')

In [10]:
import uuid
from langchain_core.documents import Document

In [37]:
docs[:5]

[Document(metadata={'page': 1, 'image_id': [], 'image_path': [], 'text_summary': [], 'image_summary': [], 'doc_id': '05328b6e-3ef9-41e6-b96a-d29000584323'}, page_content='\n\nISSUE REPORT l 2022.05.01. IS-139\n\n\nA I Index 2022의 주요 내용과 시사점\n==========================\n\nSummary and Implications of 2022 AI Index Report\n\n유재흥/조원영/안성원\n===========\n'),
 Document(metadata={'page': 2, 'image_id': [], 'image_path': [], 'text_summary': [], 'image_summary': [], 'doc_id': '43e0a290-cbb5-4f4c-882a-036072170b47'}, page_content='\n\n이 보고서는 ｢과학기술정보통신부 정보통신진흥기금｣에서 지원받아 제작한 것으로  \n과학기술정보통신부의 공식의견과 다를 수 있습니다.\n\n\n이 보고서의 내용은 연구진의 개인 견해이며, 본 보고서와 관련한 의문 사항 또는 수정·보완할  \n필요가 있는 경우에는 아래 연락처로 연락해 주시기 바랍니다.\n\n소프트웨어정책연구소\n\n유재흥 선임연구원 (jayoo@spri.kr)\n'),
 Document(metadata={'page': 3, 'image_id': [], 'image_path': [], 'text_summary': [], 'image_summary': [], 'doc_id': 'f45afb3e-9c1e-42a6-8a70-15f196d17dc6'}, page_content='\n\nSPRi 이슈리포트 IS-139\n\n\n  \nAI Index 2022의 주요 내용 및 시사점\n\nC O N T E N T\n========

In [40]:
for i,doc in enumerate(docs):
    doc.metadata["doc_id"] = str(uuid.uuid4())  
    doc.metadata["page"] = i+1

In [44]:
docs[-1]

Document(metadata={'page': 134, 'image_id': [], 'image_path': [], 'doc_id': '50cb7bbd-2256-4c25-a4bb-c6fd37197851'}, page_content='\n\nAI Index 2025의 주요 내용 및 시사점\n\n\n  \n\nSummary and Implications of 2025 AI Index Report  \n경기도 성남시 분당구 대왕판교로 712번길 22 글로벌 R&D 연구동(B) 4층\n\nGlobal R&D Center 4F 22 Daewangpangyo-ro 712beon-gil, Bundang-gu, Seongnam-si, Gyeonggi-do\n\n  \n\nwww.spri.kr\n')

In [45]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)

In [46]:
split_documents = text_splitter.split_documents(docs)

In [47]:
def insert_chunk_id(split_documents: list[Document]) -> list[Document]:
    page = 0
    for doc in split_documents:
        if page != doc.metadata["page"]:
            page = doc.metadata["page"]
            chunk_index = 1
        doc.metadata["chunk_id"] = "page_" + str(doc.metadata["page"]) + "_chunk_" + str(chunk_index)
        chunk_index += 1   
    return split_documents

In [48]:
split_documents = insert_chunk_id(split_documents)

In [50]:
split_documents[5:15]

[Document(metadata={'page': 5, 'image_id': [], 'image_path': [], 'text_summary': [], 'image_summary': [], 'doc_id': '541a20d1-b084-4ceb-96bc-386a90f8a6db', 'chunk_id': 'page_5_chunk_2'}, page_content='Recently, the Human-centered Artificial Intelligence Institute(HAI) at Stanford  \nUniversity published the AI \u200b\u200bIndex 2022. The report consists of five chapters: R&D,  \ntechnical performance, technical AI ethics, the economy and education, and AI policy  \ngovernance. In particular, this year, Chapter 2, which added detailed analysis of  \ntechnical performance of AI, and Chapter 3, which dealt with in-depth AI ethics,  \nwere newly added. According to the report, private investment in the field of AI  \ngrew to $93.5 billion in 2021, doubling compared to 2020. The United States and  \nChina are leading the R&D and startup ecosystem, and while the two countries are  \ncompeting for technological supremacy, cooperation in the research field is  \ncontinuously increasing. The pe

In [51]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5-mini", temperature=0.0)

In [52]:
DOCUMENT_CONTEXT_PROMPT = """
<document>
{doc_content}
</document>
"""
 
CHUNK_CONTEXT_PROMPT = """
Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>
 
Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else.
"""
 
 
def situate_context(docs: list[Document], chunks: list[Document]) -> dict[str, str]:
    prompts = []
    for doc in docs:
        for chunk in chunks:
            if doc.metadata['page'] == chunk.metadata['page']:
                prompt = [
                    {"role": "system", "content": "You MUST answer in Korean."},
                    {"role": "user", "content": DOCUMENT_CONTEXT_PROMPT.format(doc_content=doc.page_content)},
                    {"role": "user", "content": CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk.page_content)},
                ]
                prompts.append(prompt)
    response = llm.batch(prompts)
    return response

In [53]:
res = situate_context(docs, split_documents)

In [54]:
len(res)

248

In [55]:
for r, chunk in zip(res, split_documents):
    chunk.page_content =  r.content + "\n\n" + chunk.page_content
    chunk.metadata["contextualized_content"] = r.content + "\n\n" + chunk.page_content
    chunk.metadata["original_content"] = chunk.page_content   


In [56]:
with open(root / "outputs" / "SPRI_ALL_split_documents.pkl", "wb") as f:
    pickle.dump(split_documents, f)

In [57]:
from langchain_community.vectorstores import FAISS
from langchain_upstage import UpstageEmbeddings

embeddings = UpstageEmbeddings(model="embedding-passage")
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)
vectorstore.save_local(root / "faiss_index", "SPRI_ALL_contextual")